In [1]:
# import os
# os.environ['HUGGINGFACEHUB_API_TOKEN'] = '****************'

In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-huggingface faiss-cpu python-dotenv sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [56]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint,HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [57]:
#setup for embedding model and chat_model

#step1: embedding model
embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

#step2: chat model
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task = "text-generation"
)

chat_model = ChatHuggingFace(llm = llm)

# step 1 a - Indexing (Document Ingestion)

In [58]:
video_id = "7ARBJQn6QkM"  #enter only id , not full url
try:
  #if you don't care which language, this returns the best one , if you don't know exact language then only pass video_id
  transcript_list = YouTubeTranscriptApi().fetch(video_id,languages=['en'])

  #flatten it to plain text
  transcript = " ".join(chunk.text for chunk in transcript_list)

except TranscriptsDisabled:
  print("No Caption available for this video.")

print(transcript)

At some point, you have to believe something. 
We've reinvented computing as we know it. What is the vision for what you see coming next? We 
asked ourselves, if it can do this, how far can it go? How do we get from the robots that 
we have now to the future world that you see? Cleo, everything that moves will be 
robotic someday and it will be soon. We invested tens of billions of dollars before 
it really happened. No that's very good, you did some research! But the big breakthrough 
I would say is when we... That's Jensen Huang, and whether you know it or not
his decisions are shaping your future. He's the CEO of NVIDIA, the company that skyrocketed over the past few
years to become one of the most valuable companies in the world because they led a fundamental shift 
in how computers work unleashing this current explosion of what's possible with technology. 
"NVIDIA's done it again!" We found ourselves being one of the most important technology companies in 
the world and potentiall

#Step 1b => Indexing (Text Splitting)

In [59]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [60]:
print(len(chunks))

71


#Step 1c & 1d => Indexing (Embedding Generation and Storing in Vector Store)

In [61]:
vector_store = FAISS.from_documents(chunks,embedding_model)

In [62]:
vector_store.index_to_docstore_id

{0: '90ec900a-c5b6-47be-a8bb-269a8bdae2de',
 1: '6af17752-081c-4652-8689-80977b1c02ff',
 2: 'bd147d5f-eb39-41a5-ba7a-2c9b3dd9c5c8',
 3: 'e28cf934-f08b-4b9b-a080-4e4ac78a3497',
 4: '589091ed-8833-43ce-94a5-0930e84ee364',
 5: '5af5fb23-c917-4e2b-bfe7-5b6e0809b4bd',
 6: '0e9d1b36-8379-4430-8598-9e796bcd7206',
 7: '4c445a6c-474c-4f5e-a9eb-f22acb327faa',
 8: 'bc6e9d4f-fe75-4604-9e2b-287c41108c83',
 9: 'e7fb4885-3538-4dc6-a320-b34cae52a469',
 10: '75a4a6c6-81d1-45a5-95f8-804281489688',
 11: 'bfabb989-05bc-4847-a14e-19d51e74d378',
 12: '1d08e422-044d-4488-8ff3-7317538222ea',
 13: '922e1a51-bd38-4b1f-99e3-c99a3b090092',
 14: '1cac946f-3ec9-49a3-8fd3-4d3386d5c018',
 15: '1a8a6992-bcfa-4049-812b-3e348029ecb0',
 16: '8cd57175-c56a-46d1-9faf-9484671fd23d',
 17: 'aa1e8165-84f1-460b-b403-6e698a933b04',
 18: 'c836ba95-a074-4a51-9518-2be0dc7687c3',
 19: '6fc69a35-5201-4242-9e6a-bd442ea912e0',
 20: 'b1ed4ffc-e2b7-4ee0-82c0-1251437176f1',
 21: '3fd1eb85-064c-4272-bf13-9d2a89112d80',
 22: 'd42814b9-05ab-

In [63]:
vector_store.get_by_ids(['4d4a0bc6-0916-40a4-be3b-1400dfc10ae3'])

[]

#step 2: Retrieval

In [64]:
retriever = vector_store.as_retriever(search_type='similarity',search_kwargs={"k":4})

In [65]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7cc9bf4622a0>, search_kwargs={'k': 4})

In [66]:
retriever.invoke('what is alexnet')

[Document(id='8cd57175-c56a-46d1-9faf-9484671fd23d', metadata={}, page_content='find it someday was a good was a good strategy. It was a strategy based on hope, but it was also\xa0\nreasoned hope. The thing that really caught our attention was simultaneously we were trying\xa0\nto solve the computer vision problem inside the company and we were trying to get CUDA to\xa0\nbe a good computer vision processor and we were frustrated by a whole bunch of early\xa0\ndevelopments internally with respect to our computer vision effort and getting CUDA to be\xa0\nable to do it. And all of a sudden we saw AlexNet, this new algorithm that is completely\xa0\ndifferent than computer vision algorithms before it, take a giant leap in terms of capability\xa0\nfor computer vision. And when we saw that it was partly out of interest but partly because we were\xa0\nstruggling with something ourselves. And so we were we were highly interested to want to see it work.\xa0\nAnd so when we when we looked at Alex

#Step 3: Augmentation

In [67]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer only from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {content}\n
      Question: {question}
    """,
    input_variables=['content','question']
)

In [68]:
question = 'what is alexnet'
retrieved_docs = retriever.invoke(question)

In [69]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [70]:
final_prompt = prompt.invoke({"content":context_text,"question":question})

#Step 4: Generation

In [71]:
answer = chat_model.invoke(final_prompt)
print(answer.content)

AlexNet is a deep‑learning neural‑network architecture that was introduced in 2012. It was a breakthrough in computer vision, achieving a giant leap in performance compared with earlier algorithms, and it was trained on a huge amount of data using NVIDIA GPUs.


#Building a chain

In [72]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [76]:
def format_docs(retrieved_doc):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_doc)
  return context_text


In [85]:
parallel_chain = RunnableParallel(
    content = retriever | RunnableLambda(format_docs) ,
    question = RunnablePassthrough()
)


In [86]:
parallel_chain.invoke('what is alexnet')

{'content': "find it someday was a good was a good strategy. It was a strategy based on hope, but it was also\xa0\nreasoned hope. The thing that really caught our attention was simultaneously we were trying\xa0\nto solve the computer vision problem inside the company and we were trying to get CUDA to\xa0\nbe a good computer vision processor and we were frustrated by a whole bunch of early\xa0\ndevelopments internally with respect to our computer vision effort and getting CUDA to be\xa0\nable to do it. And all of a sudden we saw AlexNet, this new algorithm that is completely\xa0\ndifferent than computer vision algorithms before it, take a giant leap in terms of capability\xa0\nfor computer vision. And when we saw that it was partly out of interest but partly because we were\xa0\nstruggling with something ourselves. And so we were we were highly interested to want to see it work.\xa0\nAnd so when we when we looked at AlexNet we were inspired by that. But the big breakthrough I\n\nstruggl

In [87]:
parser = StrOutputParser()
main_chain = parallel_chain | prompt | chat_model | parser

In [88]:
main_chain.invoke("what is alexnet")

'AlexNet is a deep convolutional neural network that was developed for computer‑vision tasks. It achieved a dramatic improvement in image‑classification accuracy compared with earlier approaches and was the first large‑scale neural network trained efficiently on NVIDIA GPUs, demonstrating the power of GPU‑accelerated deep learning.'